## Compute a global time slice map for a chosen statistic over a given time period

In [1]:
print('Loading packages...')
import sys
sys.path.append('../00_modules/.')
from import_packages import PackageGetter
globals().update(PackageGetter.import_standard_packages_for_analysis_and_plotting())
globals().update(PackageGetter.import_custom_packages())

import xesmf as xe


Loading packages...


/g100/home/userexternal/ekoehn00/.conda/envs/eekenv/lib/python3.14/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## 0) define functions

In [2]:
def calc_vertical_stat(da, thickness_weights, stat, mask=None, dims=None):
    if stat == 'mean':
        da_stat = SpaceOperator.calc_vertical_mean(da, thickness_weights, mask=mask, dims=dims)
    elif stat == 'integral':
        da_stat = SpaceOperator.calc_vertical_integral(da, thickness_weights, mask=mask, dims=dims)
    return da_stat

def get_vert_stat(varia,default=True):
    if default == True:
        if varia in ['dissic']:
            vert_stat = 'integral'
        else:
            vert_stat = None
    else:
        vert_stat = None
    return vert_stat

In [3]:
from dask.distributed import Client
client = Client()
client
server = 'cineca'
if server == 'levante':
    port = client.scheduler_info()["services"]["dashboard"]
    prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/")
    proxy_url = f"https://jupyterhub.dkrz.de/{prefix}proxy/{port}/status"
    print(proxy_url)

## 1) get the varias, models and runs over which to do the computation

In [10]:
#stat = 'integral'#'mean'
freq_input = 'monthly'#, 'yearly', 'daily'] #freq_output = 'monthly'#, 'yearly', 'daily', 'climatology', None]
varias = ['tos','fgco2','nbp','npp','intpp','cLand','cSoil','cVeg']#'tos']#['tas','tos', #'tas', ['cLand','cSoil','cVeg']#['npp']#['cSoilFast','cSoilMedium','cSoilSlow']#['epc100','cLand','cVeg','nbp','intpp','npp']#,'fgco2',''epc100']#['epc100','fgco2','intpp','tos']#['npp','cSoil','cLand','cVeg']#['fgco2','tas']#['epc100']#['fgco2','intpp','npp','cLand']#['co2s']#['intpp','chlos','epc100']#['cLand','cVeg','cSoil','npp','co2mass','tas','tos','fgco2','nbp','fco2antt','intpp','npp','cLand','cVeg','cSoil','intdic']#['npp','cLand','fgco2','nbp','fco2antt']#'fco2antt']#['tas','nbp','fgco2','fco2antt','co2mass','tos','npp']#]#['intpp','npp','tos'] # ['fgco2','nbp','cLand','dissic','cSoil','cVeg','cLitter','cCwd']#  # 'tas',                ,'co2mass']#['fgco2']#['nbp']#['nbp','cLand'] #['nbp']#['nbp']#['fco2antt']#['cLand','cSoil','cVeg','cLitter','cCwd']#['cLitter']#'cCwd',#['cLand']#['cSoil','cVeg']#['cLitter']#['cCwd']#['cSoil','cVeg']#'cLand',#['tas','nbp','npp','tos','fgco2','intpp']
models = ['UKESM1-2']#['EC-Earth3-ESM-1'] #['NorESM2-LM']#['GISSE2.1-G-CC2']#['IPSL-CM6-ESMCO2']#['GFDL-ESM2M']#['NorESM2-LM']##['GFDL-ESM2M']#['IPSL-CM6-ESMCO2']#['CESM2']#['ACCESS-ESM1-5'] #['ACCESS-ESM1-5']#['IPSL-CM6-ESMCO2']#['NorESM2-LM']#['IPSL-CM6-ESMCO2','NorESM2-LM','GFDL-ESM2M']#['MIROC-ES2L'] ['IPSL-CM6-ESMCO2']#['EC-Earth3-ESM-1']#['UKESM1-2']#['IPSL-CM6-ESMCO2']#['IPSL-CM6-ESMCO2']#[]#['GISSE2.1-G-CC2']#['EC-Earth3-ESM-1']#,'UKESM1-2']#['NorESM2-LM']#['GFDL-ESM2M']#['IPSL-CM6-ESMCO2']#['UKESM1-2']#['IPSL-CM6-ESMCO2','NorESM2-LM','GFDL-ESM2M'] # ,
runs = pruns.get_run_list('tipmip_tier1')#[:1]#[5:]#[1:]#[:-1]#[:1]#[-1:]#[:1]#[:1]#[1:]#[-1:] #

# choose the type of temporal statisic
temporal_stats = ['mean'] # 'std', 'linear trend', 
# choose the time window length (in years) for rolling_mean
rm_window = 31 # in years
# choose the time_window_size for tim slices
time_window_size = 21 # in years
# choose the reference 
reference = 'first_year_of_rampup' # 'first_year_of_piC', 'first_10y_of_rampup', ...
# choose GMST levels at which to calculate time slice maps (for rampup and rampdown). For the piC, and stabilizations, I calculate the time slice statistic over the first time_window_size years
#gmst_levels = [0,1,2,3,4]

server='cineca'#'levante'
outroot = './../01_postprocessed_data/global_time_slice_statistic_maps/'

# Define the time slice names
time_slice_names = ['piC_at_start',
                    'rampup_1K',
                    'rampup_2K',
                    'rampup_3K',
                    'rampup_4K',
        
                    'stab2K',
                    'stab4K',
        
                    'rampdn2K_1K',
                    'rampdn2K_0K',
        
                    'rampdn4K_3K',
                    'rampdn4K_2K',     
                    'rampdn4K_1K',
                    'rampdn4K_0K',             
        
                    'restab2K']

def identify_run(time_slice_name):
    if 'rampup' in time_slice_name:
        run = 'esm-up2p0'
    elif 'piC' in time_slice_name:
        run = 'esm-piControl'
    elif time_slice_name == 'stab2K':
        run = 'esm-up2p0-gwl2p0'
    elif time_slice_name == 'stab4K':
        run = 'esm-up2p0-gwl4p0'        
    elif 'rampdn2K' in time_slice_name:
        run = 'esm-up2p0-gwl2p0-50y-dn2p0'
    elif 'rampdn4K' in time_slice_name:
        run = 'esm-up2p0-gwl4p0-50y-dn2p0'
    elif time_slice_name == 'restab2K':
        run = 'esm-up2p0-gwl4p0-50y-dn2p0-gwl2p0' 
    return run


def get_time_range_underlying_time_slice(model,time_slice_name,reference,time_window_size,rm_window,centered_running_mean=True):    #,running_mean_years=31,centered_running_mean=True,reference_slice='jones2025'):

    # get the global mean surface temperature
    varia = 'tas'
    run = identify_run(time_slice_name)

    # get the model_dict for extra info
    model_dict = pmods.get_model_dict('all')
    
    # Decide what to do
    if '_0K' in time_slice_name or '_1K' in time_slice_name or '_2K' in time_slice_name or '_3K' in time_slice_name or '_4K' in time_slice_name:
        print('    ... need to identify the time range')
        need_to_identify = True
    elif 'stab' in time_slice_name or time_slice_name == 'piC_at_start':
        print('    ... no need to identify anything - already clearly defined')
        need_to_identify = False
    
    # If there is a need to identify:
    if need_to_identify:

        # Handle the reference data
        if reference == 'first_year_of_rampup':
            reference_run = 'esm-up2p0'
        elif reference == 'first_year_of_piC':
            reference_run = 'esm-piControl'
            
        # get the reference data
        print(f'    ... loading in the GMST data for the reference run {reference_run}.')
        ref_dir = f'./../01_postprocessed_data/global_time_series/{varia}/{model}/{reference_run}/{member}/{freq_input}/global_mean'
        ref_str = f'{ref_dir}/{varia}_{model}_{reference_run}_{member}_global_mean.nc'
        print(f'    ... loading reference: {ref_str}')
        with xr.open_dataset(ref_str,use_cftime=True) as ref_ds:
            if 'first_year' in reference:
                ref_ds = TimeOperator.shift_time_axis_by_n_years(ref_ds,n=0)
                ref_ds_annual = ref_ds.resample(time='1YS').mean()
                t0 = cftime.DatetimeProlepticGregorian(model_dict[model].rampup_start_year,1,1)
                ref_val = ref_ds_annual[f'{varia}_global_mean'].sel(time=t0).values 
            #print(ref_val)
        
        # Handle the run data
        print(f'    ... loading in the GMST data for the run {run}.')
        run_dir = f'./../01_postprocessed_data/global_time_series/{varia}/{model}/{run}/{member}/{freq_input}/global_mean'
        run_str = f'{run_dir}/{varia}_{model}_{run}_{member}_global_mean.nc'
        print(f'    ... loading run: {run_str}')
        with xr.open_dataset(run_str,use_cftime=True) as run_ds:
            if 'first_year' in reference:
                run_ds_annual = run_ds.resample(time='1YS').mean()
                run_ds_annual_anom = run_ds_annual[f'{varia}_global_mean'] - ref_val

                # do a rolling_mean
                run_ds_annual_anom_rm = run_ds_annual_anom.rolling(time=rm_window,center=centered_running_mean,min_periods=1).mean(dim='time')
                
                # get the GWL level
                GWL = int(time_slice_name.split('_')[-1][0])
                print(f'    ... GWL = {GWL}K')
                
                # masks
                if run == 'esm-up2p0':
                    mask_bool = run_ds_annual_anom_rm > GWL                    
                elif run in ['esm-up2p0-gwl2p0-50y-dn2p0','esm-up2p0-gwl4p0-50y-dn2p0']:
                    mask_bool = run_ds_annual_anom_rm < GWL                    

                
                if mask_bool.any(dim="time"):
                    first_idx = mask_bool.argmax(dim="time")
                    first_time = run_ds_annual_anom_rm.time.isel(time=first_idx)
                    first_year = int(first_time.dt.year.values)
        
                    print(
                        f'    ... first crossing at anomaly of '
                        f'{run_ds_annual_anom_rm.isel(time=first_idx).values}K'
                    )
                    
                    time_slice_start_year = int(first_year - (time_window_size-1)/2)
                    time_slice_end_year = int(first_year + (time_window_size-1)/2)
                else:
                    first_year = np.nan
                    time_slice_start_year = np.nan
                    time_slice_end_year = np.nan               


    else:
        print(f'    ... no need for any reference run.')
        if time_slice_name == 'piC_at_start':
            time_slice_start_year = model_dict[model].rampup_start_year
            time_slice_end_year = model_dict[model].rampup_start_year+time_window_size-1 
        elif time_slice_name == 'stab2K':
            time_slice_start_year = model_dict[model].stab2K_start_year
            time_slice_end_year = model_dict[model].stab2K_start_year+50 - time_window_size+1 #  last (time_window_size)years of the stabilization before rampdown
        elif time_slice_name == 'stab4K':
            time_slice_start_year = model_dict[model].stab4K_start_year
            time_slice_end_year = model_dict[model].stab4K_start_year+50 - time_window_size+1 #  last (time_window_size)years of the stabilization before rampdown
        elif time_slice_name == 'restab2K':
            time_slice_start_year = model_dict[model].restab2K_start_year
            time_slice_end_year = model_dict[model].restab2K_start_year+50 - time_window_size+1  #  last (time_window_size)years of the restabilization

    # turn nan years into None
    print(time_slice_start_year)
    print(time_slice_end_year)
    if np.isnan(time_slice_start_year):
        time_slice_start_year = None
    if np.isnan(time_slice_end_year):
        time_slice_end_year = None
    print(time_slice_start_year)
    print(time_slice_end_year)
    
    return time_slice_start_year, time_slice_end_year


def calc_time_slice_stat(varia,model,time_slice_name,time_slice_start_year,time_slice_end_year,temporal_stat='mean'):

    if model == 'CESM2':
        print(varia)
        varia = mgrab.varia_mapper_cmor_to_model(varia)
        print(varia)
                
    # first get the dataset
    da = mgrab.get_data(varia,run,freq_input=freq_input,verbose_level=0)#,server=server)#,server=server)
    print(f'... loading {da.time.size} data points in time.')

    if model == 'CESM2' and varia == 'TEMP':
        print('choosing only the shallowest layer')
        da = da.isel(z_t=0)#.squeeze().persist()
        print(da)

    # make sure the time dimension is Proleptic Gregorian
    da = TimeOperator.shift_time_axis_by_n_years(da,n=0)
    
    # get unit
    unit = da.units
    
    # resample to annual means
    da_annual = da.resample(time='1YS').mean(dim='time') # weight with month lengths?
    #print(da_annual.time)

    # cut out the time slice
    #da_annual_slice = da_annual.sel(time=slice(cftime.DatetimeProlepticGregorian(time_slice_start_year,1,1),cftime.DatetimeProlepticGregorian(time_slice_end_year,1,1)))

    if time_slice_start_year is None or time_slice_end_year is None:
    
        da_annual_slice = xr.full_like(
            da_annual.isel(time=slice(0, 2)).astype(float),
            np.nan,
        )
    
    else:
        da_annual_slice = da_annual.sel(
            time=slice(
                cftime.DatetimeProlepticGregorian(time_slice_start_year, 1, 1),
                cftime.DatetimeProlepticGregorian(time_slice_end_year, 1, 1),
            )
        )

    
    # If the variable is npp or nbp, weight with the land area fraction
    grid_cell_fractions = mgrab.get_area_fraction(varia) # not directly used, but required for writing some attributes later on

    # take into account that coastal cells are not 100% land
    if grid_cell_fractions is not None:
        print('adjust area weights for coastal points')
        assert np.max(grid_cell_fractions.values)<=1
        assert np.min(grid_cell_fractions.values)>=0
        # make sure that the coordinates area the same
        for coord in da_annual_slice.coords:
            if coord == 'time':
                continue
            diff_coord = grid_cell_fractions[coord].values - da_annual_slice[coord].values
            sum_diff_coord = np.sum(np.abs(diff_coord))
            unique_diffs = np.unique(diff_coord)
            if sum_diff_coord > 0:
                print(f'There is a total coordinate difference of: {sum_diff_coord}')
                if np.sum(np.abs((grid_cell_fractions[coord].values - da_annual_slice[coord].values))) < 1e-1:
                    print(f'... this is small enough (smaller than 1e-1), so we just set the grid_cell_fractions.coord to area_weights.coord.')
                    grid_cell_fractions = grid_cell_fractions.assign_coords({coord: da_annual_slice[coord]})
                elif np.all(np.isin(unique_diffs, [-360, 0, 360])):
                    print(f'... all of the differences are just caused by 360° longitude wrapping. So we just set the grid_cell_fractions.coord to area_weights.coord.')
                    grid_cell_fractions = grid_cell_fractions.assign_coords({coord: da_annual_slice[coord]})        
                else:
                    raise Exception('Coordinates do not match')
        # now multiply them together
        da_annual_slice = da_annual_slice * grid_cell_fractions.values
        multiplied_with_area_fraction = True
    else:
        multiplied_with_area_fraction = False

    if temporal_stat == 'mean':
        temp_statistic = da_annual_slice.mean(dim='time')
        unit = unit
    else:
        raise Exception('This temporal_stat is not yet defined.')

    print(temp_statistic)
    
    return temp_statistic, unit, multiplied_with_area_fraction


def regrid_field(ds,tmp_in,tmp_out,method='xesmf'):

    #print(ds)

    if method == 'cdo':
        subprocess.run(["cdo", "remapdis,r360x180", tmp_in, tmp_out],check=True)
        ds_out = xr.open_dataset(tmp_out)
        ds_out.attrs.update(global_attrs)
        ds_out.to_netcdf(tmp_out+'2', mode="w")
        subprocess.run(["mv", tmp_out+'2', tmp_out], check=True)
        return print('done with cdo')

    elif method == 'xesmf':

        #print('REGRIDDING')
        #print(ds)

        # Make a copy to avoid modifying original dataset
        ds_copy = ds.copy()

        
        ## Case 1: 2D lat/lon coordinates exist
        if "geolat_t" in ds_copy.coords and "geolon_t" in ds_copy.coords:
            print("geolat_t/geolon_t present")
            
            ds_copy = ds_copy.set_coords(["geolat_t", "geolon_t"])
            ds_copy = ds_copy.rename({"geolat_t": "lat", "geolon_t": "lon"})
            # Load into memory if dask arrays
            ds_copy["lat"] = ds_copy["lat"].load()
            ds_copy["lon"] = ds_copy["lon"].load()
            # Optional: add CF attributes
            ds_copy["lat"].attrs.update({
                "standard_name": "latitude",
                "long_name": "Latitude of T points",
                "units": "degrees_north"
            })
            ds_copy["lon"].attrs.update({
                "standard_name": "longitude",
                "long_name": "Longitude of T points",
                "units": "degrees_east"
            })
    
        # Target grid: r360x180
        target = xr.Dataset({
            "lat": (["lat"], np.linspace(-89.5, 89.5, 180)),
            "lon": (["lon"], np.linspace(-179.5, 179.5, 360)),
        })
    
        # Create the regridder
        regridder = xe.Regridder(ds_copy, target, method="nearest_s2d", periodic=True)
    
        # Regrid the dataset
        ds_regridded = regridder(ds_copy)
    
        # Update global attributes and save
        #ds_regridded.attrs.update(global_attrs)

        #print('SAVING')
        #ds_regridded.to_netcdf(tmp_out)
        print('        ... done with xesmf')

    return ds_regridded # 


def regrid_field(ds, tmp_in, tmp_out, method='xesmf'):

    import xarray as xr
    import numpy as np
    import xesmf as xe
    import subprocess

    # ==========================================================
    # CDO METHOD
    # ==========================================================
    if method == 'cdo':

        subprocess.run(
            ["cdo", "remapdis,r360x180", tmp_in, tmp_out],
            check=True
        )

        ds_out = xr.open_dataset(tmp_out)

        # Optional global attrs
        # ds_out.attrs.update(global_attrs)

        ds_out.to_netcdf(tmp_out + '2', mode="w")

        subprocess.run(
            ["mv", tmp_out + '2', tmp_out],
            check=True
        )

        print('done with cdo')

        return ds_out

    # ==========================================================
    # XESMF METHOD
    # ==========================================================
    elif method == 'xesmf':

        # ------------------------------------------
        # Handle both DataArray and Dataset inputs
        # ------------------------------------------
        input_is_dataarray = isinstance(ds, xr.DataArray)

        if input_is_dataarray:
            varname = ds.name or "var"
            ds_copy = ds.to_dataset(name=varname)
        else:
            ds_copy = ds.copy()

        # ------------------------------------------
        # Handle MOM-style curvilinear grids
        # ------------------------------------------
        if "geolat_t" in ds_copy.coords and "geolon_t" in ds_copy.coords:

            print("geolat_t/geolon_t present")

            ds_copy = ds_copy.set_coords(["geolat_t", "geolon_t"])

            ds_copy = ds_copy.rename({
                "geolat_t": "lat",
                "geolon_t": "lon"
            })

            # Ensure coordinates are loaded
            ds_copy["lat"] = ds_copy["lat"].load()
            ds_copy["lon"] = ds_copy["lon"].load()

            # Add CF-compliant attributes
            ds_copy["lat"].attrs.update({
                "standard_name": "latitude",
                "long_name": "Latitude",
                "units": "degrees_north"
            })

            ds_copy["lon"].attrs.update({
                "standard_name": "longitude",
                "long_name": "Longitude",
                "units": "degrees_east"
            })

        # ------------------------------------------
        # Target regular 1-degree grid
        # ------------------------------------------
        target = xr.Dataset({
            "lat": (
                ["lat"],
                np.linspace(-89.5, 89.5, 180)
            ),
            "lon": (
                ["lon"],
                np.linspace(-179.5, 179.5, 360)
            ),
        })

        # ------------------------------------------
        # Create regridder
        # ------------------------------------------
        regridder = xe.Regridder(
            ds_copy,
            target,
            method="nearest_s2d",
            periodic=True
        )

        # ------------------------------------------
        # Regrid
        # ------------------------------------------
        ds_regridded = regridder(ds_copy)

        # ------------------------------------------
        # Optional global attrs
        # ------------------------------------------
        # ds_regridded.attrs.update(global_attrs)

        # ------------------------------------------
        # Convert back to DataArray if needed
        # ------------------------------------------
        if input_is_dataarray:
            ds_regridded = ds_regridded[varname]

        print('        ... done with xesmf')

        return ds_regridded

In [11]:
for varia in varias:
    for model in models:

        mgrab = MODELgrabber.get_grabber(model)
        #stat = get_stat(varia,model)
        member = mgrab.get_member()
        
        print(f'---{model}---')
        temporal_stat_dict = dict()
        for time_slice_name in time_slice_names:

            if time_slice_name == 'restab2K' and model in ['GISSE2.1-G-CC2','EC-Earth3-ESM-1']:
                continue

            # identify the run for the time slice statistic
            run = identify_run(time_slice_name)
            
            print(f'-> calculate time slice statistic(s) for {varia}, {model}, {time_slice_name}: in {run}.')

            print('... identify the time range over which I want to calculate the time slice.')
            time_slice_start_year, time_slice_end_year = get_time_range_underlying_time_slice(model,time_slice_name,reference,time_window_size,rm_window,centered_running_mean=True)
            print('... the time range for the time slice statistic is going to be:',time_slice_start_year, time_slice_end_year)

            print('... compute the statisic(s)')
            for temporal_stat in temporal_stats:

                print(f'    ... computing the {temporal_stat}...')
                statistic,unit,multiplied_with_area_fraction = calc_time_slice_stat(varia,model,time_slice_name,time_slice_start_year,time_slice_end_year,temporal_stat=temporal_stat)
                
                print(f'    ... regrid the {temporal_stat} to a 1x1 degree grid...')
                #global_attrs = statistic.attrs()
                tmp_in = os.path.join("./", "in.nc")
                tmp_out = os.path.join("./", "out.nc")
                statistic_regridded = regrid_field(statistic,tmp_in,tmp_out,method='xesmf')
                statistic_regridded.attrs["unit"] = unit
                statistic_regridded.attrs["time_slice_start_year"] = (np.nan if time_slice_start_year is None else time_slice_start_year)
                statistic_regridded.attrs["time_slice_end_year"] = (np.nan if time_slice_end_year is None else time_slice_end_year)
                
                temporal_stat_dict[f'{time_slice_name}_{temporal_stat}'] = statistic_regridded
                print(f' ')

        temporal_stat_ds = xr.Dataset(temporal_stat_dict)
        # add global attributes to dataset
        temporal_stat_ds.attrs = {
            "title": f"Global time slice {temporal_stat}s",
            "model": f"{model}",
            "member": f"{member}",
            "varia": f"{varia}",
            "freq_input": f"{freq_input}",
            "temporal_stat": f"{temporal_stat}",
            "rm_window for identifying GWL crossing": f"{rm_window}",
            "time_window_size": f"{time_window_size}",
            "reference": f"{reference}",
            "outroot": f"{outroot}",
            "unit": f"{unit}",
            "multiplied_with_area_fraction": f"{multiplied_with_area_fraction}",
            "author": "E. E. Köhn",
            "created": "2026-05-20",
        }
        # save the dataset 
        outdir = f"{outroot}/{varia}/{model}/ref_{reference}_win{time_window_size}yr/{temporal_stat}"
        os.makedirs(outdir, exist_ok=True)
        outfile = f"{outdir}/{varia}_{model}_{temporal_stat}_ref_{reference}_win{time_window_size}yr.nc"
        temporal_stat_ds.to_netcdf(outfile)
        

---UKESM1-2---
-> calculate time slice statistic(s) for tos, UKESM1-2, piC_at_start: in esm-piControl.
... identify the time range over which I want to calculate the time slice.
    ... no need to identify anything - already clearly defined
    ... no need for any reference run.
1850
1870
1850
1870
... the time range for the time slice statistic is going to be: 1850 1870
... compute the statisic(s)
    ... computing the mean...
/g100_store/DRES_OptimESM/ESGF/prepub/mohc/2*/CMIP6/CMIP/MOHC/UKESM1-2/esm-piControl/r1i1p1f1/Omon/tos/gn/v*/tos*_gn_*.nc


sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found


... loading 4812 data points in time.
<xarray.DataArray 'tos' (j: 330, i: 360)> Size: 475kB
dask.array<mean_agg-aggregate, shape=(330, 360), dtype=float32, chunksize=(330, 360), chunktype=numpy.ndarray>
Coordinates:
  * j          (j) int32 1kB 0 1 2 3 4 5 6 7 ... 322 323 324 325 326 327 328 329
  * i          (i) int32 1kB 0 1 2 3 4 5 6 7 ... 352 353 354 355 356 357 358 359
    latitude   (j, i) float64 950kB dask.array<chunksize=(330, 360), meta=np.ndarray>
    longitude  (j, i) float64 950kB dask.array<chunksize=(330, 360), meta=np.ndarray>
Attributes:
    standard_name:  sea_surface_temperature
    long_name:      Sea Surface Temperature
    comment:        Temperature of upper boundary of the liquid ocean, includ...
    units:          degC
    original_name:  mo: (variable_name: tos)
    cell_methods:   area: mean where sea time: mean
    cell_measures:  area: areacello
    ... regrid the mean to a 1x1 degree grid...


sh: getfattr: command not found


        ... done with xesmf
 
-> calculate time slice statistic(s) for tos, UKESM1-2, rampup_1K: in esm-up2p0.
... identify the time range over which I want to calculate the time slice.
    ... need to identify the time range
    ... loading in the GMST data for the reference run esm-up2p0.
    ... loading reference: ./../01_postprocessed_data/global_time_series/tas/UKESM1-2/esm-up2p0/r1i1p1f1/monthly/global_mean/tas_UKESM1-2_esm-up2p0_r1i1p1f1_global_mean.nc


...

sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found


<xarray.DataArray 'cVeg' (lat: 144, lon: 192)> Size: 111kB
dask.array<mean_agg-aggregate, shape=(144, 192), dtype=float32, chunksize=(144, 192), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float64 1kB -89.38 -88.12 -86.88 -85.62 ... 86.88 88.12 89.38
  * lon      (lon) float64 2kB 0.9375 2.812 4.688 6.562 ... 355.3 357.2 359.1
Attributes:
    standard_name:  vegetation_carbon_content
    long_name:      Carbon Mass in Vegetation
    comment:        Carbon mass per unit area in vegetation.
    units:          kg m-2
    original_name:  mo: (stash: m01s19i002, lbtim_ia: 240, lbproc: 128)
    cell_methods:   area: mean where land time: mean
    cell_measures:  area: areacella
    history:        2025-01-17T00:57:05Z altered by CMOR: replaced missing va...
    ... regrid the mean to a 1x1 degree grid...
        ... done with xesmf
 
-> calculate time slice statistic(s) for cVeg, UKESM1-2, restab2K: in esm-up2p0-gwl4p0-50y-dn2p0-gwl2p0.
... identify the time range over which I 

sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found


... loading 1944 data points in time.
<xarray.DataArray 'cVeg' (lat: 144, lon: 192)> Size: 111kB
dask.array<mean_agg-aggregate, shape=(144, 192), dtype=float32, chunksize=(144, 192), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float64 1kB -89.38 -88.12 -86.88 -85.62 ... 86.88 88.12 89.38
  * lon      (lon) float64 2kB 0.9375 2.812 4.688 6.562 ... 355.3 357.2 359.1
Attributes:
    standard_name:  vegetation_carbon_content
    long_name:      Carbon Mass in Vegetation
    comment:        Carbon mass per unit area in vegetation.
    units:          kg m-2
    original_name:  mo: (stash: m01s19i002, lbtim_ia: 240, lbproc: 128)
    cell_methods:   area: mean where land time: mean
    cell_measures:  area: areacella
    history:        2026-03-04T12:46:35Z altered by CMOR: replaced missing va...
    ... regrid the mean to a 1x1 degree grid...
        ... done with xesmf
 


sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getf